In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import logging
import json
from pathlib import Path

import pandas as pd
from transformers import GenerationConfig

from pivotal_tokens.hf.loading import load_model, load_tokenizer
from pivotal_tokens.hf.sampling import batch_sample, extract_thinking_trace
from pivotal_tokens.hf.dataset import load_hotpotqa_dataset
from pivotal_tokens.extractor import SuccessProbabilityShiftExtractor
from pivotal_tokens.oracle import RegexOracle
from pivotal_tokens.repo import DictRepo
from pivotal_tokens.utils import setup_logging

In [3]:
HF_MODEL_ID = "Qwen/Qwen3-1.7B"
HF_DATASET_ID = "hotpotqa/hotpot_qa"
DICT_REPO_DIR = Path("repo")
ORACLE_ANSWER_REGEX = r"(?s)<\|im_start\|>assistant.*?(?:</think>\s*)?Answer:\s*(.*?)\s*(?=(?:<\|im_end\|>|<\|endoftext\|>|\Z))"

SYSTEM_PROMPT = ("Answer the following question directly, strictly without chain-of-thought after \"</think>\"."
                 "Keep the answer short (e.g., \"yes\" or \"no\" for binary questions, a person's full name if "
                 "the question expects a person, a country name if it asks about a country, etc.). Output the "
                 "answer strictly after the prefix \"Answer: \"  with no extra text.")

In [4]:
setup_logging(logging.DEBUG)

# model = load_model(model_id=HF_MODEL_ID, device="cuda")

In [5]:
# tokenizer = load_tokenizer(model_id=HF_MODEL_ID)


# generation_config = GenerationConfig(temperature=0.6,
#                                      top_p=0.95,
#                                      top_k=20,
#                                      min_p=0.1,
#                                      max_new_tokens=4096,
#                                      num_return_sequences=1,
#                                      do_sample=True)

oracle = RegexOracle(answer_regex=ORACLE_ANSWER_REGEX,
                        fuzzy_match_threshold=0.8)


samples = load_hotpotqa_dataset(name="fullwiki", split="validation")
samples = samples[:30]

# completions = batch_sample(samples=samples,
#                            system_prompt=SYSTEM_PROMPT,
#                            model=model,
#                            tokenizer=tokenizer,
#                            generation_config=generation_config,
#                            batch_size=8)

# traces = [extract_thinking_trace(c) for c in completions]

2025-12-09 01:09:27,241 - urllib3.connectionpool - DEBUG - Starting new HTTPS connection (1): huggingface.co:443
2025-12-09 01:09:27,387 - urllib3.connectionpool - DEBUG - https://huggingface.co:443 "HEAD /datasets/hotpotqa/hotpot_qa/resolve/main/README.md HTTP/1.1" 307 0
2025-12-09 01:09:27,402 - urllib3.connectionpool - DEBUG - https://huggingface.co:443 "HEAD /api/resolve-cache/datasets/hotpotqa/hotpot_qa/1908d6afbbead072334abe2965f91bd2709910ab/README.md HTTP/1.1" 200 0
2025-12-09 01:09:27,527 - urllib3.connectionpool - DEBUG - https://huggingface.co:443 "HEAD /datasets/hotpotqa/hotpot_qa/resolve/1908d6afbbead072334abe2965f91bd2709910ab/hotpot_qa.py HTTP/1.1" 404 0
2025-12-09 01:09:27,533 - urllib3.connectionpool - DEBUG - Starting new HTTPS connection (1): s3.amazonaws.com:443
2025-12-09 01:09:28,152 - urllib3.connectionpool - DEBUG - https://s3.amazonaws.com:443 "HEAD /datasets.huggingface.co/datasets/datasets/hotpotqa/hotpot_qa/hotpotqa/hotpot_qa.py HTTP/1.1" 404 0
2025-12-09 01

In [ ]:
# oracle_outputs = []
# for s, c in zip(samples, completions):
#     success = oracle.verify(actual=c, expected=[s.ground_truth])
#     oracle_outputs.append(int(success))

In [7]:
# df = pd.DataFrame({"id": [s.id for s in samples],
#               "query": [s.query for s in samples],
#               "ground_truth": [s.ground_truth for s in samples],
#               "trace": traces,
#               "completion": completions,
#               "is_correct": oracle_results,
#               "metadata": [json.dumps(s.metadata) for s in samples]})

In [ ]:
# !mkdir -p data
# df.to_csv("data/hotpotqa_fullwiki_qwen3_1.7B_results.csv", index=False)

In [20]:
df = pd.read_csv("data/hotpotqa_fullwiki_qwen3_1.7B_results.csv")
df[:2]

,id,query,ground_truth,trace,completion,is_correct,metadata
0,5a8b57f25542995d1e6f1371,Were Scott Derrickson and Ed Wood of the same ...,yes,"<think>\nOkay, let's see. The question is whet...",<|endoftext|><|endoftext|><|endoftext|><|endof...,1,"{""id"": ""5a8b57f25542995d1e6f1371"", ""question"":..."
1,5a8c7595554299585d9e36b6,What government position was held by the woman...,Chief of Protocol,"<think>\nOkay, let's see. The question is aski...",<|endoftext|><|endoftext|><|endoftext|><|endof...,0,"{""id"": ""5a8c7595554299585d9e36b6"", ""question"":..."
